In [1]:
import pandas as pd

In [2]:
df3 = pd.read_csv('Dataset.csv')

In [4]:
df3.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [5]:
df3.isnull().sum()

,0
show_id,0
type,0
title,0
director,0
country,0
date_added,0
release_year,0
rating,0
duration,0
listed_in,0


In [3]:
print(df3['rating'].value_counts())

rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64


In [6]:
print("\nNumber of unique rating categories:", df3['rating'].nunique())


Number of unique rating categories: 14


In [8]:
from sklearn.preprocessing import LabelEncoder

features3 = df3[['type', 'director', 'country', 'listed_in', 'release_year']].copy()
target3 = df3['rating']

encoders3 = {}
for col in ['type', 'director', 'country', 'listed_in']:
    le = LabelEncoder()
    features3[col] = le.fit_transform(features3[col])
    encoders3[col] = le

target_le = LabelEncoder()
target3_encoded = target_le.fit_transform(target3)


print("\nRating classes:", list(target_le.classes_))


Rating classes: ['G', 'NC-17', 'NR', 'PG', 'PG-13', 'R', 'TV-14', 'TV-G', 'TV-MA', 'TV-PG', 'TV-Y', 'TV-Y7', 'TV-Y7-FV', 'UR']


In [9]:
print(features3.head())

   type  director  country  listed_in  release_year
0     0      2294       80        273          2020
1     1      2104       20        241          2021
2     1      2865       80        498          2021
3     0       627        6        124          2021
4     0      1503       80        318          1993


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

X_train3, X_test3, y_train3, y_test3 = train_test_split(
    features3, target3_encoded, test_size=0.2, random_state=42
)

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train3, y_train3)

rf_model3 = RandomForestClassifier(random_state=42)
rf_model3.fit(X_train3, y_train3)

print("Models trained successfully")

Models trained successfully


In [11]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train3, y_train3)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation accuracy:", grid_search.best_score_)

best_rf_model3 = grid_search.best_estimator_

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(


Best parameters: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 100}
Best cross-validation accuracy: 0.5172070534698521


In [15]:
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
# Compare Decision Tree, base Random Forest, and tuned Random Forest
dt_test_acc = accuracy_score(y_test3, dt_model.predict(X_test3))
rf_test_acc = accuracy_score(y_test3, rf_model3.predict(X_test3))
best_rf_test_acc = accuracy_score(y_test3, best_rf_model3.predict(X_test3))
labels_present = np.unique(np.concatenate([y_test3, best_rf_model3.predict(X_test3)]))
names_present = target_le.inverse_transform(labels_present)
print("Decision Tree Test Accuracy:", round(dt_test_acc, 3))
print("Random Forest Test Accuracy:", round(rf_test_acc, 3))
print("Tuned Random Forest Test Accuracy:", round(best_rf_test_acc, 3))


Decision Tree Test Accuracy: 0.44
Random Forest Test Accuracy: 0.506
Tuned Random Forest Test Accuracy: 0.544

Tuned Random Forest Classification Report:


In [17]:
import numpy as np

labels_present = np.unique(np.concatenate([y_test3, best_rf_model3.predict(X_test3)]))
names_present = target_le.inverse_transform(labels_present)
print("\nTuned Random Forest Classification Report:")
print(classification_report(
    y_test3,
    best_rf_model3.predict(X_test3),
    labels=labels_present,
    target_names=names_present,
    zero_division=0
))


Tuned Random Forest Classification Report:
              precision    recall  f1-score   support

           G       0.50      0.08      0.14        12
          NR       0.00      0.00      0.00        21
          PG       0.55      0.48      0.51        54
       PG-13       0.50      0.30      0.38        93
           R       0.46      0.50      0.48       155
       TV-14       0.60      0.49      0.54       444
        TV-G       0.57      0.09      0.16        44
       TV-MA       0.55      0.82      0.66       628
       TV-PG       0.50      0.11      0.18       181
        TV-Y       0.56      0.58      0.57        66
       TV-Y7       0.44      0.54      0.49        59
          UR       0.00      0.00      0.00         1

    accuracy                           0.54      1758
   macro avg       0.44      0.33      0.34      1758
weighted avg       0.54      0.54      0.51      1758

